In [1]:
# Data handling
import pandas as pd
import numpy as np

# Load saved ML objects
import joblib

# For nicer tables
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
from pathlib import Path
import joblib
import pandas as pd

PROJECT_ROOT = Path("..")

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
MODEL_PATH = PROJECT_ROOT / "models"
OUTPUT_PATH = PROJECT_ROOT / "outputs"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
MODEL_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
import joblib

PROJECT_PATH = "/content/drive/MyDrive/AI-Relationship-Manager"

# Load trained XGBoost model
xgboost_model = joblib.load(
    MODEL_PATH / "xgboost_model.pkl"
)

preprocessor = joblib.load(
    MODEL_PATH / "preprocessor.pkl"
)

print("Model loaded successfully!")
print("Preprocessor loaded successfully!")

Model loaded successfully!
Preprocessor loaded successfully!


In [ ]:
# Load the cleaned customer dataset
model_df = pd.read_csv(
    PROCESSED_PATH / "cleaned_customer_data.csv"
)

print("Dataset loaded successfully!")
print()

print("Shape :", model_df.shape)
model_df.head()

Dataset loaded successfully!

Shape : (7043, 20)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes


In [9]:
# Separate input features from the target column

X = model_df.drop("Churn Label", axis=1)

print("Feature matrix created successfully!")
print()

print("Shape:", X.shape)
X.head()

Feature matrix created successfully!

Shape: (7043, 19)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30


In [10]:
# Transform the raw customer data using the saved preprocessing pipeline

X_processed = preprocessor.transform(X)

print("Data preprocessed successfully!")
print()

print("Processed Shape :", X_processed.shape)
print("Data Type :", type(X_processed))

Data preprocessed successfully!

Processed Shape : (7043, 46)
Data Type : <class 'numpy.ndarray'>


In [11]:
# Predict churn (0 = No, 1 = Yes)

predictions = xgboost_model.predict(X_processed)

print("Predictions generated successfully!")

print("\nPrediction Shape:", predictions.shape)

print("\nFirst 10 Predictions:")
print(predictions[:10])

Predictions generated successfully!

Prediction Shape: (7043,)

First 10 Predictions:
[0 1 1 0 0 0 1 0 0 1]


In [12]:
# Predict churn probability for every customer

probabilities = xgboost_model.predict_proba(X_processed)

print("Probabilities generated successfully!")

print()

print("Shape :", probabilities.shape)

print()

print("First 5 probability predictions:")

print(probabilities[:5])

Probabilities generated successfully!

Shape : (7043, 2)

First 5 probability predictions:
[[0.58042073 0.41957927]
 [0.37200564 0.62799436]
 [0.14409298 0.855907  ]
 [0.78131884 0.21868117]
 [0.76674944 0.23325056]]


In [13]:
# Extract only churn probability

churn_probability = probabilities[:, 1]

print("Churn probabilities extracted successfully!")

print()

print("Shape :", churn_probability.shape)

print()

print("First 10 churn probabilities:")

print(churn_probability[:10])

Churn probabilities extracted successfully!

Shape : (7043,)

First 10 churn probabilities:
[0.41957927 0.62799436 0.855907   0.21868117 0.23325056 0.2672481
 0.931474   0.4708119  0.08922093 0.6663487 ]


In [14]:
# Create the AI Relationship Manager dataset

results_df = model_df.copy()

# Add the AI prediction
results_df["Prediction"] = predictions

# Add churn probability (convert to percentage)
results_df["Churn Probability (%)"] = (
    churn_probability * 100
).round(2)

print("Results table created successfully!")

print()

results_df.head()

Results table created successfully!



,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Prediction,Churn Probability (%)
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0,41.959999
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,62.799999
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,85.589996
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,0,21.870001
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,0,23.330000


In [15]:
# Categorize customers into business-friendly risk levels

def assign_risk(probability):

    if probability >= 80:
        return "Very High"

    elif probability >= 60:
        return "High"

    elif probability >= 40:
        return "Medium"

    else:
        return "Low"


# Apply the function to every customer
results_df["Risk Level"] = results_df["Churn Probability (%)"].apply(assign_risk)

print("Risk levels assigned successfully!")

print()

# Check the first few customers
results_df[
    ["Prediction", "Churn Probability (%)", "Risk Level"]
].head(10)

Risk levels assigned successfully!



,Prediction,Churn Probability (%),Risk Level
0,0,41.959999,Medium
1,1,62.799999,High
2,1,85.589996,Very High
3,0,21.870001,Low
4,0,23.330000,Low
5,0,26.719999,Low
6,1,93.150002,Very High
7,0,47.080002,Medium
8,0,8.920000,Low
9,1,66.629997,High


In [ ]:
# Load SHAP explanations generated in Notebook 05

shap_df = pd.read_csv(
    PROCESSED_PATH / "shap_values.csv"
)

feature_df = pd.read_csv(
    PROCESSED_PATH / "feature_names.csv"
)

print("SHAP values loaded successfully!")

print()

print("SHAP Shape :", shap_df.shape)

print()

print("Feature Names :", feature_df.shape)

SHAP values loaded successfully!

SHAP Shape : (1409, 46)

Feature Names : (46, 1)


In [18]:
# Enterprise Knowledge Base

knowledge_base = {

    "Contract_Month-to-month": {
        "reason": "Customer has a flexible month-to-month contract and may have lower long-term commitment.",
        "recommendation": "Offer a discounted annual contract with additional benefits.",
        "priority": "High",
        "department": "Retention Team"
    },

    "Contract_One year": {
        "reason": "Customer already has a medium-term commitment.",
        "recommendation": "Offer loyalty rewards before contract renewal.",
        "priority": "Medium",
        "department": "Customer Success"
    },

    "Contract_Two year": {
        "reason": "Customer is already highly committed.",
        "recommendation": "Maintain engagement through loyalty benefits.",
        "priority": "Low",
        "department": "Customer Success"
    },

    "Tenure Months": {
        "reason": "Short-tenure customers usually have weaker loyalty.",
        "recommendation": "Assign onboarding specialist and welcome offers.",
        "priority": "High",
        "department": "Retention Team"
    },

    "Monthly Charges": {
        "reason": "Higher monthly charges can increase churn risk.",
        "recommendation": "Provide a loyalty discount or recommend a better-value plan.",
        "priority": "Medium",
        "department": "Sales"
    },

    "Total Charges": {
        "reason": "Higher lifetime value indicates an important customer.",
        "recommendation": "Prioritize personalized retention efforts.",
        "priority": "Medium",
        "department": "Retention Team"
    },

    "Online Security_No": {
        "reason": "Customer is not subscribed to Online Security.",
        "recommendation": "Offer a complimentary Online Security trial.",
        "priority": "High",
        "department": "Sales"
    },

    "Online Backup_No": {
        "reason": "Customer does not use Online Backup services.",
        "recommendation": "Bundle Online Backup into the existing package.",
        "priority": "Medium",
        "department": "Sales"
    },

    "Device Protection_No": {
        "reason": "Customer lacks Device Protection coverage.",
        "recommendation": "Promote Device Protection with a limited-time discount.",
        "priority": "Medium",
        "department": "Sales"
    },

    "Tech Support_No": {
        "reason": "Customer is not subscribed to Tech Support.",
        "recommendation": "Provide complimentary Tech Support for a limited period.",
        "priority": "Medium",
        "department": "Customer Success"
    },

    "Internet Service_Fiber optic": {
        "reason": "Fiber customers generally have higher service expectations.",
        "recommendation": "Offer premium customer support or network health review.",
        "priority": "Medium",
        "department": "Technical Support"
    },

    "Internet Service_DSL": {
        "reason": "Customer uses DSL internet.",
        "recommendation": "Introduce upgraded broadband packages if available.",
        "priority": "Low",
        "department": "Sales"
    },

    "Payment Method_Electronic check": {
        "reason": "Electronic check users historically exhibit higher churn.",
        "recommendation": "Encourage automatic bank or card payments.",
        "priority": "Low",
        "department": "Finance"
    },

    "Paperless Billing_Yes": {
        "reason": "Customer prefers digital communication.",
        "recommendation": "Send personalized digital loyalty offers.",
        "priority": "Low",
        "department": "Marketing"
    },

    "Partner_No": {
        "reason": "Customer does not have a partner account.",
        "recommendation": "Promote family or multi-user plans.",
        "priority": "Low",
        "department": "Sales"
    },

    "Dependents_No": {
        "reason": "Customer has no dependents linked to the account.",
        "recommendation": "Offer bundled household packages.",
        "priority": "Low",
        "department": "Sales"
    },

    "Multiple Lines_No": {
        "reason": "Customer only uses a single phone line.",
        "recommendation": "Promote multi-line discounts.",
        "priority": "Low",
        "department": "Sales"
    },

    "Senior Citizen_Yes": {
        "reason": "Senior customers may benefit from additional assistance.",
        "recommendation": "Assign priority customer support and simplified plans.",
        "priority": "Medium",
        "department": "Customer Success"
    }

}

print("Enterprise Knowledge Base Created!")
print(f"Total Rules: {len(knowledge_base)}")

Enterprise Knowledge Base Created!
Total Rules: 18


In [19]:
# Function to identify the most influential SHAP features

def get_top_features(customer_index, top_n=5):

    # Get SHAP values for one customer
    customer_shap = shap_df.iloc[customer_index]

    # Convert to absolute values so we measure importance
    importance = customer_shap.abs()

    # Sort from highest impact to lowest
    top_features = importance.sort_values(
        ascending=False
    ).head(top_n)

    return top_features.index.tolist()


print("Function created successfully!")

Function created successfully!


In [22]:
import shap
import joblib

explainer = shap.TreeExplainer(xgboost_model)


print("SHAP Explainer loaded successfully!")

SHAP Explainer loaded successfully!


In [23]:
def generate_customer_health_report(customer_index, top_n=5):
    """
    Generate a complete AI Relationship Manager report
    for a single customer.
    """
    customer = X.iloc[[customer_index]]

    # Preprocess customer
    customer_processed = preprocessor.transform(customer)

    # Prediction
    prediction = xgboost_model.predict(customer_processed)[0]

    # Probability
    probability = xgboost_model.predict_proba(customer_processed)[0][1]

    # SHAP values
    shap_values = explainer.shap_values(customer_processed)

    # Convert SHAP values into a Series
    shap_series = pd.Series(
        shap_values[0],
        index=feature_names
    )

    # Get top contributing features
    top_features = (
        shap_series.abs()
        .sort_values(ascending=False)
        .head(top_n)
    )

    recommendations = []

    # Match features with knowledge base
    for feature in top_features.index:

        if feature in knowledge_base:

            recommendations.append({

                "Feature": feature,

                "Reason":
                knowledge_base[feature]["reason"],

                "Recommendation":
                knowledge_base[feature]["recommendation"],

                "Priority":
                knowledge_base[feature]["priority"],

                "Department":
                knowledge_base[feature]["department"]

            })

    return {

        "Prediction":
        "Churn" if prediction == 1 else "Stay",

        "Probability":
        round(probability * 100, 2),

        "Top Features":
        top_features,

        "Recommendations":
        pd.DataFrame(recommendations)

    }

print("Customer Health Report function created successfully!")

Customer Health Report function created successfully!


In [25]:
feature_names = feature_df["Feature"].tolist()

print("Feature names loaded successfully!")

print(f"Total Features: {len(feature_names)}")

print("\nFirst 10 Features:")
print(feature_names[:10])

Feature names loaded successfully!
Total Features: 46

First 10 Features:
['Tenure Months', 'Monthly Charges', 'Total Charges', 'Gender_Female', 'Gender_Male', 'Senior Citizen_No', 'Senior Citizen_Yes', 'Partner_No', 'Partner_Yes', 'Dependents_No']


In [26]:
report = generate_customer_health_report(0)

print("Prediction:")
print(report["Prediction"])

print("\nProbability:")
print(report["Probability"])

print("\nTop SHAP Features:")
print(report["Top Features"])

print("\nRecommendations:")
report["Recommendations"]

Prediction:
Stay

Probability:
41.96

Top SHAP Features:
Contract_Month-to-month         0.555945
Tenure Months                   0.527879
Online Security_No              0.356954
Internet Service_Fiber optic    0.235185
Total Charges                   0.217589
dtype: float32

Recommendations:


,Feature,Reason,Recommendation,Priority,Department
0,Contract_Month-to-month,Customer has a flexible month-to-month contrac...,Offer a discounted annual contract with additi...,High,Retention Team
1,Tenure Months,Short-tenure customers usually have weaker loy...,Assign onboarding specialist and welcome offers.,High,Retention Team
2,Online Security_No,Customer is not subscribed to Online Security.,Offer a complimentary Online Security trial.,High,Sales
3,Internet Service_Fiber optic,Fiber customers generally have higher service ...,Offer premium customer support or network heal...,Medium,Technical Support
4,Total Charges,Higher lifetime value indicates an important c...,Prioritize personalized retention efforts.,Medium,Retention Team


In [27]:
# Health Score = 100 - Churn Probability

results_df["Health Score"] = (
    100 - results_df["Churn Probability (%)"]
).round(2)

print("Health Score calculated successfully!")

results_df[
    ["Churn Probability (%)", "Health Score"]
].head(10)

Health Score calculated successfully!


,Churn Probability (%),Health Score
0,41.959999,58.040001
1,62.799999,37.200001
2,85.589996,14.410000
3,21.870001,78.129997
4,23.330000,76.669998
5,26.719999,73.279999
6,93.150002,6.850000
7,47.080002,52.919998
8,8.920000,91.080002
9,66.629997,33.369999


In [28]:
def get_health_status(score):
    """
    Convert Health Score into an easy-to-understand status.
    """

    if score >= 80:
        return "🟢 Healthy"

    elif score >= 60:
        return "🟡 Stable"

    elif score >= 40:
        return "🟠 Needs Attention"

    else:
        return "🔴 Critical"


results_df["Health Status"] = (
    results_df["Health Score"]
    .apply(get_health_status)
)

print("Health Status assigned successfully!")

results_df[
    ["Health Score", "Health Status"]
].head(10)

Health Status assigned successfully!


,Health Score,Health Status
0,58.040001,🟠 Needs Attention
1,37.200001,🔴 Critical
2,14.410000,🔴 Critical
3,78.129997,🟡 Stable
4,76.669998,🟡 Stable
5,73.279999,🟡 Stable
6,6.850000,🔴 Critical
7,52.919998,🟠 Needs Attention
8,91.080002,🟢 Healthy
9,33.369999,🔴 Critical


In [36]:
feature_translator = {

    "Contract_Month-to-month":
        "Customer is on a month-to-month contract.",

    "Contract_One year":
        "Customer has a one-year contract.",

    "Contract_Two year":
        "Customer has a two-year contract.",

    "Tenure Months":
        "Customer has relatively short tenure.",

    "Monthly Charges":
        "Customer has high monthly charges.",

    "Total Charges":
        "Customer has high lifetime value.",

    "Online Security_No":
        "Customer has not subscribed to Online Security.",

    "Online Backup_No":
        "Customer does not use Online Backup.",

    "Device Protection_No":
        "Customer does not have Device Protection.",

    "Tech Support_No":
        "Customer is not subscribed to Tech Support.",

    "Internet Service_Fiber optic":
        "Customer uses Fiber Optic internet.",

    "Internet Service_DSL":
        "Customer uses DSL internet.",

    "Payment Method_Electronic check":
        "Customer pays using Electronic Check.",

    "Partner_No":
        "Customer does not have a partner.",

    "Dependents_No":
        "Customer has no dependents.",

    "Multiple Lines_No":
        "Customer has a single phone line.",

    "Paperless Billing_Yes":
        "Customer uses paperless billing.",

    "Senior Citizen_Yes":
        "Customer is a senior citizen."

}

print("Feature Translator created successfully!")

Feature Translator created successfully!


In [43]:
# Generate Complete AI Customer Report

def generate_ai_customer_report(customer_index):

    # Generate the detailed health report
    report = generate_customer_health_report(customer_index)

    # Fetch customer details
    customer = results_df.iloc[customer_index]

    print("=" * 60)
    print("        AI RELATIONSHIP MANAGER")
    print("=" * 60)

    print(f"\nCustomer Index : {customer_index}")

    print(f"\nPrediction : {report['Prediction']}")

    print(f"Churn Probability : {report['Probability']:.2f}%")

    print(f"Health Score : {customer['Health Score']:.2f}/100")

    print(f"Health Status : {customer['Health Status']}")

    print(f"Risk Level : {customer['Risk Level']}")

    print("\n" + "=" * 60)
    print("Top Churn Drivers")
    print("=" * 60)

    for feature in report["Top Features"].index:

      if feature in feature_translator:
          print(f"• {feature_translator[feature]}")
      else:
          print(f"• {feature}")

    print("\n" + "=" * 60)
    print("Recommended Actions")
    print("=" * 60)

    recommendation_table = report["Recommendations"].copy()

    # Replace technical feature names with business-friendly text
    recommendation_table["Feature"] = (
        recommendation_table["Feature"]
        .apply(lambda x: feature_translator.get(x, x))
    )
    display(recommendation_table)

    print("\n" + "=" * 60)
    print("EXECUTIVE SUMMARY")
    print("=" * 60)

    summary = generate_executive_summary(
      customer,
      report
    )

    print(summary)

    return report


print("AI Customer Report Generator created successfully!")

AI Customer Report Generator created successfully!


In [38]:
generate_ai_customer_report(0)

        AI RELATIONSHIP MANAGER

Customer Index : 0

Prediction : Stay
Churn Probability : 41.96%
Health Score : 58.04/100
Health Status : 🟠 Needs Attention
Risk Level : Medium

Top Churn Drivers
• Customer is on a month-to-month contract.
• Customer has relatively short tenure.
• Customer has not subscribed to Online Security.
• Customer uses Fiber Optic internet.
• Customer has high lifetime value.

Recommended Actions


,Feature,Reason,Recommendation,Priority,Department
0,Contract_Month-to-month,Customer has a flexible month-to-month contrac...,Offer a discounted annual contract with additi...,High,Retention Team
1,Tenure Months,Short-tenure customers usually have weaker loy...,Assign onboarding specialist and welcome offers.,High,Retention Team
2,Online Security_No,Customer is not subscribed to Online Security.,Offer a complimentary Online Security trial.,High,Sales
3,Internet Service_Fiber optic,Fiber customers generally have higher service ...,Offer premium customer support or network heal...,Medium,Technical Support
4,Total Charges,Higher lifetime value indicates an important c...,Prioritize personalized retention efforts.,Medium,Retention Team


{'Prediction': 'Stay',
 'Probability': np.float32(41.96),
 'Top Features': Contract_Month-to-month         0.555945
 Tenure Months                   0.527879
 Online Security_No              0.356954
 Internet Service_Fiber optic    0.235185
 Total Charges                   0.217589
 dtype: float32,
 'Recommendations':                         Feature  \
 0       Contract_Month-to-month   
 1                 Tenure Months   
 2            Online Security_No   
 3  Internet Service_Fiber optic   
 4                 Total Charges   
 
                                               Reason  \
 0  Customer has a flexible month-to-month contrac...   
 1  Short-tenure customers usually have weaker loy...   
 2     Customer is not subscribed to Online Security.   
 3  Fiber customers generally have higher service ...   
 4  Higher lifetime value indicates an important c...   
 
                                       Recommendation Priority  \
 0  Offer a discounted annual contract with additi..

Feature Translator created successfully!


In [39]:
# Generate Executive Summary

def generate_executive_summary(customer, report):

    summary = f"""
Customer Health Summary

The customer is currently predicted to **{report['Prediction']}**
with a churn probability of **{report['Probability']:.2f}%**.

Current Health Score:
{customer['Health Score']:.2f}/100

Health Status:
{customer['Health Status']}

Risk Level:
{customer['Risk Level']}

The primary churn drivers include:

"""

    for feature in report["Top Features"].index:

        if feature in feature_translator:
            summary += f"\n• {feature_translator[feature]}"

    summary += """

Recommended Strategy

• Contact customer proactively.
• Prioritize the recommendations shown above.
• Monitor customer engagement over the next billing cycle.

"""

    return summary


print("Executive Summary function created successfully!")

Executive Summary function created successfully!


In [44]:
generate_ai_customer_report(0)

        AI RELATIONSHIP MANAGER

Customer Index : 0

Prediction : Stay
Churn Probability : 41.96%
Health Score : 58.04/100
Health Status : 🟠 Needs Attention
Risk Level : Medium

Top Churn Drivers
• Customer is on a month-to-month contract.
• Customer has relatively short tenure.
• Customer has not subscribed to Online Security.
• Customer uses Fiber Optic internet.
• Customer has high lifetime value.

Recommended Actions


,Feature,Reason,Recommendation,Priority,Department
0,Customer is on a month-to-month contract.,Customer has a flexible month-to-month contrac...,Offer a discounted annual contract with additi...,High,Retention Team
1,Customer has relatively short tenure.,Short-tenure customers usually have weaker loy...,Assign onboarding specialist and welcome offers.,High,Retention Team
2,Customer has not subscribed to Online Security.,Customer is not subscribed to Online Security.,Offer a complimentary Online Security trial.,High,Sales
3,Customer uses Fiber Optic internet.,Fiber customers generally have higher service ...,Offer premium customer support or network heal...,Medium,Technical Support
4,Customer has high lifetime value.,Higher lifetime value indicates an important c...,Prioritize personalized retention efforts.,Medium,Retention Team



EXECUTIVE SUMMARY

Customer Health Summary

The customer is currently predicted to **Stay**
with a churn probability of **41.96%**.

Current Health Score:
58.04/100

Health Status:
🟠 Needs Attention

Risk Level:
Medium

The primary churn drivers include:


• Customer is on a month-to-month contract.
• Customer has relatively short tenure.
• Customer has not subscribed to Online Security.
• Customer uses Fiber Optic internet.
• Customer has high lifetime value.

Recommended Strategy

• Contact customer proactively.
• Prioritize the recommendations shown above.
• Monitor customer engagement over the next billing cycle.




{'Prediction': 'Stay',
 'Probability': np.float32(41.96),
 'Top Features': Contract_Month-to-month         0.555945
 Tenure Months                   0.527879
 Online Security_No              0.356954
 Internet Service_Fiber optic    0.235185
 Total Charges                   0.217589
 dtype: float32,
 'Recommendations':                         Feature  \
 0       Contract_Month-to-month   
 1                 Tenure Months   
 2            Online Security_No   
 3  Internet Service_Fiber optic   
 4                 Total Charges   
 
                                               Reason  \
 0  Customer has a flexible month-to-month contrac...   
 1  Short-tenure customers usually have weaker loy...   
 2     Customer is not subscribed to Online Security.   
 3  Fiber customers generally have higher service ...   
 4  Higher lifetime value indicates an important c...   
 
                                       Recommendation Priority  \
 0  Offer a discounted annual contract with additi..

In [45]:
# Estimate annual revenue at risk
results_df["Annual Revenue at Risk"] = (
    results_df["Monthly Charges"] * 12
).round(2)

print("Annual Revenue at Risk calculated successfully!")

results_df[
    [
        "Monthly Charges",
        "Annual Revenue at Risk"
    ]
].head()

Annual Revenue at Risk calculated successfully!


,Monthly Charges,Annual Revenue at Risk
0,53.85,646.2
1,70.70,848.4
2,99.65,1195.8
3,104.80,1257.6
4,103.70,1244.4


In [48]:
def assign_priority(row):

    if row["Health Score"] < 40 and row["Annual Revenue at Risk"] >= 1000:
        return "Critical"

    elif row["Health Score"] < 60:
        return "High"

    elif row["Health Score"] < 80:
        return "Medium"

    else:
        return "Low"


results_df["Customer Priority"] = results_df.apply(
    assign_priority,
    axis=1
)

print("Customer Priority assigned successfully!")

results_df[
    [
        "Health Score",
        "Annual Revenue at Risk",
        "Customer Priority"
    ]
].head(10)

Customer Priority assigned successfully!


,Health Score,Annual Revenue at Risk,Customer Priority
0,58.040001,646.2,High
1,37.200001,848.4,High
2,14.410000,1195.8,Critical
3,78.129997,1257.6,Medium
4,76.669998,1244.4,Medium
5,73.279999,662.4,Medium
6,6.850000,475.8,High
7,52.919998,241.8,High
8,91.080002,1192.2,Low
9,33.369999,362.4,High


In [ ]:
output_path = OUTPUT_PATH / "ai_relationship_manager_output.csv"

results_df.to_csv(
    output_path,
    index=False
)

print("Customer Health Report saved successfully!")
print(output_path)

Customer Health Report saved successfully!
/content/drive/MyDrive/AI-Relationship-Manager/outputs/ai_relationship_manager_output.csv


In [50]:
# Load the saved file to verify

final_df = pd.read_csv(output_path)

print(final_df.shape)

final_df.head()

(7043, 27)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Prediction,Churn Probability (%),Risk Level,Health Score,Health Status,Annual Revenue at Risk,Customer Priority
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0,41.96,Medium,58.04,🟠 Needs Attention,646.2,High
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,62.80,High,37.20,🔴 Critical,848.4,High
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,85.59,Very High,14.41,🔴 Critical,1195.8,Critical
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,0,21.87,Low,78.13,🟡 Stable,1257.6,Medium
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,0,23.33,Low,76.67,🟡 Stable,1244.4,Medium
